# Deletion / Insertion Benchmark

Evaluate saliency map quality using the deletion and insertion metrics from [Petsiuk et al. (2018)](https://arxiv.org/abs/1806.07421).

- **Deletion**: Progressively remove the most salient pixels and measure prediction drop. Lower AUC = better explanation.
- **Insertion**: Progressively reveal the most salient pixels on a blurred image. Higher AUC = better explanation.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

from causal_explainer import SFL, RelevanceScore, CausalMetric, auc, get_device
from causal_explainer.benchmark.evaluation import blur_image
from causal_explainer.utils import read_tensor

In [ ]:
device = get_device()
print(f'Using device: {device}')

model = models.resnet50(pretrained=True)
model = nn.Sequential(model, nn.Softmax(dim=1))
model = model.eval().to(device)
for p in model.parameters():
    p.requires_grad = False

In [ ]:
# Generate explanation for one image
img_path = '../data_demo/catdog.png'
img_tensor = read_tensor(img_path).to(device)

with torch.no_grad():
    output = model(img_tensor)
    target_class = output.argmax(dim=1).item()

N = 200
sfl = SFL(model, input_size=(224, 224))
masks, sampled_tensor = sfl.generate_mutants_batch(img_path, N, s=8, p1=0.2, target_class=target_class)

confidence_scores = np.zeros(N)
with torch.no_grad():
    for j in range(N):
        single_image = sampled_tensor[j].unsqueeze(0)
        output = model(single_image)
        confidence_scores[j] = torch.max(output, dim=1).values.item() * 100

rs = RelevanceScore()
pixel_dataset, ochiai, tarantula, zoltar, wong1 = rs.run(confidence_scores, sampled_tensor, masks, N)

In [ ]:
# Run deletion metric
deletion = CausalMetric(model, 'del', 224, substrate_fn=torch.zeros_like, device=device)
del_scores = deletion.single_run(img_tensor, ochiai, verbose=1)
print(f'Deletion AUC (Ochiai): {auc(del_scores):.4f}')

In [ ]:
# Run insertion metric
insertion = CausalMetric(model, 'ins', 224, substrate_fn=blur_image, device=device)
ins_scores = insertion.single_run(img_tensor, ochiai, verbose=1)
print(f'Insertion AUC (Ochiai): {auc(ins_scores):.4f}')

In [ ]:
# Compare all 4 formulas
formulas = {'Ochiai': ochiai, 'Tarantula': tarantula, 'Zoltar': zoltar, 'Wong-1': wong1}

for name, explanation in formulas.items():
    del_scores = deletion.single_run(img_tensor, explanation)
    ins_scores = insertion.single_run(img_tensor, explanation)
    print(f'{name:12s} | Deletion AUC: {auc(del_scores):.4f} | Insertion AUC: {auc(ins_scores):.4f}')